 # `huggingface-hub` API  
We add here code snippets and comments for all APIs and modules of `huggingface-hib` library.

## HF credential management
Main reference is [HF tutorial on huggingface-hub](https://huggingface.co/docs/huggingface_hub/en/quick-start#manage-multiple-tokens-locally)

1. Shared token files (all huggingface-hub functions use these files)

In [ ]:
%%bash

# "Weird" assignment: := sets HF_HOME to the passed value if unset. We use the "no-command" : of bash to prevent printing
: ${HF_HOME:=~/.cache/huggingface}

echo "${HF_HOME}/token"
cat "${HF_HOME}/token"; echo; echo

echo "${HF_HOME}/stored_tokens"
cat "${HF_HOME}/stored_tokens"

# show the name of the currently logged in user
echo $(huggingface-cli whoami)

In [28]:
from huggingface_hub import login, whoami, HFCacheInfo, HfFolder, auth_list, auth_switch, auth_check

In [ ]:
import os

# Convenience store
# 1. readonly token
rotoken = os.environ["read-only-token"]
# 2. fine-grained token
fgtoken = os.environ["fine-graoned-token"]

! huggingface-cli login --token {fgtoken}

In [16]:
# Login to HF using a token literal
login(token=rotoken)

# Now the 'token' file contains the selected token. this token will be used by HF libraries globaly on this computer!
# to disable this behavior (i.e. make HF ask for authentication each time) set HF_HUB_DISABLE_IMPLICIT_TOKEN=1
print(HfFolder.get_token() == rotoken)

# Check the names of the global functions in HfFolder (only )
print(*[o for o in dir(HfFolder) if not o.startswith('__')])

True
delete_token get_token save_token


In [29]:
# There's no direct way to get the name of the currently selected token.
# Indirectly, use whoami:
print(whoami()["auth"]["accessToken"]["displayName"])

# To get all the token ever registered with HF use the auth_list function or huggingface-cli
print(auth_list())
!huggingface-cli auth list

fine-grained-token
  name                | token          
----------------------|---------------
* fine-grained-token  | hf_****KBKQ    
  read-token          | hf_****nHdC    
None
  name                | token          
----------------------|---------------
* fine-grained-token  | hf_****KBKQ    
  read-token          | hf_****nHdC    


In [40]:
# To switch to another token (by name) again use auth_switch() or the cli
# There's no command for printing the current selection. Instead, a * is placed on the left

# run huggingface-cli auth switch --help for other options
auth_switch(token_name="fine-grained-token")
print(auth_list())

!huggingface-cli auth switch --token-name read-token

  name                | token          
----------------------|---------------
* fine-grained-token  | hf_****KBKQ    
  read-token          | hf_****nHdC    
None
Your token has been saved to /Users/christos/.cache/huggingface/token
The current active token is: read-token


## Repository Management

In [ ]:
from huggingface_hub import HfApi, ModelInfo
from huggingface_hub import list_models

type(next(list_models(model_name="deepseek-ai/DeepSeek-R1-0528-Qwen3-8B")))
print(ModelInfo(id="deepseek-ai/DeepSeek-R1-0528-Qwen3-8B").json)

ModelInfo(id='deepseek-ai/DeepSeek-R1-0528-Qwen3-8B', author=None, sha=None, created_at=None, last_modified=None, private=None, disabled=None, downloads=None, downloads_all_time=None, gated=None, gguf=None, inference=None, inference_provider_mapping=None, likes=None, library_name=None, tags=None, pipeline_tag=None, mask_token=None, card_data=None, widget_data=None, model_index=None, config=None, transformers_info=None, trending_score=None, siblings=None, spaces=None, safetensors=None, security_repo_status=None, xet_enabled=None)


In [ ]:
# Download an HF repo (model) snapshot
from huggingface_hub import hf_hub_download, snapshot_download
from os.path import expanduser, join

# Example: download a small model to a custom dir
snapshot_download(
    # Pick a relatively small model at chance
    repo_id="azizdh00/MNLP_M3_rag_model",
    # Quite some files will be added here (model.safetensors, tokenizer.json, vocab.json, tokenizer_config.json, README.md,...)
    # the directory is created if it doesn't exist
    local_dir="local-files",
    # These are not the same as the folders and files that are cached
    # The repo is added here, with the folder structure described in 
    cache_dir=expanduser("~/.cache/chached-files")
)

Fetching 12 files: 100%|██████████| 12/12 [00:02<00:00,  4.28it/s]


'/Users/christos/projects/remotes/public/vllm/var/local-files'